# TP2 — Module Transformers & LLMs : Classification Ordinale sur LABR
## Groupe 3 : Amine Bouaouda & Mohammed El Amrani
### Partie A : Audit du Corpus LABR, Prétraitement et Modèles Baselines (TF-IDF + Régression Logistique)

**Objectif de la tâche :** Prédire la note attribuée par un lecteur (1 à 5 étoiles) à partir de sa critique de livre en arabe sur Goodreads (**dataset LABR** — Aly & Atiya, ACL 2013).
**Spécificité :** Il s'agit d'une **classification ordinale** : se tromper entre 4 et 5 étoiles est nettement moins grave que de se tromper entre 1 et 5 étoiles.
**Métriques imposées :** **Macro-F1** (métrique principale), **MAE** (Mean Absolute Error sur les étoiles) et **F1 pondéré**.
> *L'Accuracy est expressément interdite par le sujet car trompeuse sur un corpus fortement déséquilibré.*

In [13]:
import os
import sys
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import pyarabic.araby as araby

# Seaborn optionnel
try:
    import seaborn as sns
except ImportError:
    sns = None

# Racine du projet pour métriques
sys.path.append(os.path.abspath("../.."))
from src.models.labr_metrics import compute_labr_metrics
from src.preprocess.labr_text import clean_arabic_text

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')


## 1. Chargement du dataset LABR et Découpage Officiel
Conformément à la consigne du Groupe 3, nous utilisons le split officiel 5 classes déséquilibré :
- `reviews.tsv` (63 257 critiques au total)
- `5class-unbalanced-train.txt` (50 606 index pour l'entraînement)
- `5class-unbalanced-test.txt` (12 651 index pour le test final, scellé jusqu'à la Partie F)
- **Validation :** 10% stratifié extrait du train officiel (45 545 train, 5 061 val).

In [14]:
# 1. Détection automatique du chemin vers data/labr
data_dir = "../../data/labr" if os.path.exists("../../data/labr") else ("data/labr" if os.path.exists("data/labr") else "../data/labr")
reviews_path = os.path.join(data_dir, "reviews.tsv")
train_ids_path = os.path.join(data_dir, "5class-unbalanced-train.txt")
test_ids_path = os.path.join(data_dir, "5class-unbalanced-test.txt")

# 2. Lecture directe de reviews.tsv avec Pandas
print("1/3 Lecture directe de reviews.tsv...")
df_full = pd.read_csv(
    reviews_path,
    sep="\t",
    header=None,
    names=["rating", "review_id", "user_id", "book_id", "body"],
    encoding="utf-8",
    quoting=3,
    on_bad_lines="skip"
)

# Nettoyage des valeurs nulles et conversion des types
df_full["rating"] = pd.to_numeric(df_full["rating"], errors="coerce")
df_full = df_full.dropna(subset=["rating", "body"])
df_full["rating"] = df_full["rating"].astype(int)

# 3. Chargement direct des index officiels (50 606 train et 12 651 test)
print("2/3 Chargement des index officiels train et test...")
with open(train_ids_path, "r", encoding="utf-8") as f:
    train_indices = [int(line.strip()) for line in f if line.strip()]
with open(test_ids_path, "r", encoding="utf-8") as f:
    test_indices = [int(line.strip()) for line in f if line.strip()]

df_train_official = df_full.iloc[train_indices].copy().reset_index(drop=True)
df_test_official = df_full.iloc[test_indices].copy().reset_index(drop=True)

# 4. Découpage officiel Train (90%) / Val (10% stratifié)
train_df, val_df = train_test_split(
    df_train_official,
    test_size=0.10,
    random_state=RANDOM_STATE,
    stratify=df_train_official["rating"]
)

print("3/3 Dataset LABR chargé avec succès !")
print(f" -> Total critiques brutes : {len(df_full):,}")
print(f" -> Train officiel (90%)   : {len(train_df):,} critiques")
print(f" -> Val officiel (10%)     : {len(val_df):,} critiques")
print(f" -> Test officiel (scellé) : {len(df_test_official):,} critiques")


1/3 Lecture directe de reviews.tsv...
2/3 Chargement des index officiels train et test...
3/3 Dataset LABR chargé avec succès !
 -> Total critiques brutes : 63,257
 -> Train officiel (90%)   : 45,545 critiques
 -> Val officiel (10%)     : 5,061 critiques
 -> Test officiel (scellé) : 12,651 critiques


## 2. 4.1 Audit des Données (Critiques GoodReads)
Analyse de la distribution des notes, longueurs en mots, tranches imposées et doublons.

In [15]:
# 1. Distribution des étoiles (1 à 5)
dist = train_df['rating'].value_counts().sort_index()
dist_pct = train_df['rating'].value_counts(normalize=True).sort_index() * 100
print("Distribution des notes dans le Train officiel :")
print(pd.DataFrame({'Nombre': dist, 'Pourcentage (%)': dist_pct.round(2)}))

# 2. Longueurs des critiques en mots
train_df['length_words'] = train_df['body'].str.split().str.len()
val_df['length_words'] = val_df['body'].str.split().str.len()

print("\nStatistiques de longueur des critiques (en mots) :")
print(train_df['length_words'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

# 3. Tranches de longueur imposées par la fiche Groupe 3
bins = [0, 10, 50, 150, 500, 100000]
labels = ['Très courte (<10)', 'Courte (10-50)', 'Moyenne (50-150)', 'Longue (150-500)', 'Très longue (>500)']
train_df['tranche_longueur'] = pd.cut(train_df['length_words'], bins=bins, labels=labels)
val_df['tranche_longueur'] = pd.cut(val_df['length_words'], bins=bins, labels=labels)

print("\nTaux de chaque note (%) par tranche de longueur :")
cross_len = pd.crosstab(train_df['tranche_longueur'], train_df['rating'], normalize='index') * 100
print(cross_len.round(2))

# 4. Doublons
exact_dup = train_df.duplicated(subset=['body']).sum()
norm_dup = train_df['body'].apply(clean_arabic_text).duplicated().sum()
print(f"\nDoublons exacts de texte dans Train : {exact_dup}")
print(f"Doublons après normalisation dans Train : {norm_dup}")

Distribution des notes dans le Train officiel :
        Nombre  Pourcentage (%)
rating                         
1         2103             4.62
2         3777             8.29
3         8857            19.45
4        13694            30.07
5        17114            37.58

Statistiques de longueur des critiques (en mots) :
count    45545.000000
mean        62.104117
std        108.578389
min          1.000000
5%           4.000000
25%         14.000000
50%         32.000000
75%         67.000000
95%        213.000000
max       3419.000000
Name: length_words, dtype: float64

Taux de chaque note (%) par tranche de longueur :
rating                 1     2      3      4      5
tranche_longueur                                   
Très courte (<10)   6.00  7.08  15.61  26.84  44.48
Courte (10-50)      4.50  8.70  19.41  29.87  37.52
Moyenne (50-150)    4.08  8.62  21.65  32.00  33.65
Longue (150-500)    3.59  7.87  22.11  32.91  33.52
Très longue (>500)  5.65  7.32  17.15  29.08  40.79

Doubl

## 3. Prétraitement et Normalisation du Texte Arabe
Conformément aux consignes de la fiche du TP2 pour le Groupe 3, nous étudions :
1. La présence de **tatweel** (ـ) et de **diacritiques** (tashkeel).
2. La présence de **mots non-arabes / caractères latins** et de **critiques très courtes**.
3. L'implémentation du pipeline de nettoyage textuel adapté à l'arabe littéraire (Goodreads).
4. L'évaluation de l'impact du prétraitement par ablation.

In [16]:
# Audit des particularités linguistiques et textuelles
has_tatweel = train_df['body'].str.contains(r'ـ').sum()
has_tashkeel = train_df['body'].apply(lambda t: bool(re.search(r'[ً-ْ]', str(t)))).sum()
has_latin = train_df['body'].str.contains(r'[a-zA-Z]').sum()
is_tiny = (train_df['length_words'] <= 2).sum()

print('--- Audit Linguistique des Critiques (Train) ---')
print(f'Critiques avec Tatweel (ـ)         : {has_tatweel:,} ({has_tatweel/len(train_df)*100:.2f}%)')
print(f'Critiques avec Tashkeel (harakat)   : {has_tashkeel:,} ({has_tashkeel/len(train_df)*100:.2f}%)')
print(f'Critiques avec mots/caract. latins  : {has_latin:,} ({has_latin/len(train_df)*100:.2f}%)')
print(f'Critiques quasi-vides (<= 2 mots)   : {is_tiny:,} ({is_tiny/len(train_df)*100:.2f}%)')

--- Audit Linguistique des Critiques (Train) ---
Critiques avec Tatweel (ـ)         : 2,287 (5.02%)
Critiques avec Tashkeel (harakat)   : 14,130 (31.02%)
Critiques avec mots/caract. latins  : 0 (0.00%)
Critiques quasi-vides (<= 2 mots)   : 1,124 (2.47%)


In [17]:
def preprocess_arabic_pipeline(text, remove_repeated=True):
    if not isinstance(text, str):
        return ''
    # 1. Élimination du tatweel (kashida)
    text = araby.strip_tatweel(text)
    # 2. Élimination des diacritiques (harakat : fatha, damma, kasra, tanween, sukun, shadda)
    text = araby.strip_tashkeel(text)
    # 3. Normalisation des alifs (أ, إ, آ -> ا)
    text = re.sub(r'[إأآا]', 'ا', text)
    # 4. Normalisation de l'alif maqsura vers yaa (ى -> ي)
    text = re.sub(r'ى', 'ي', text)
    # 5. Normalisation de la taa marbuta vers haa (ة -> ه)
    text = re.sub(r'ة', 'ه', text)
    # 6. Compression des allongements expressifs (> 2 répétitions, ex: رووووعة -> روعة)
    if remove_repeated:
        text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    # 7. Normalisation des espaces et sauts de ligne
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Démonstration Avant / Après sur 3 critiques réelles
print('--- Exemples de Prétraitement (Avant vs Après) ---')
samples = train_df[train_df['body'].str.contains(r'ـ|[إأآ]')].head(3)
for idx, row in samples.iterrows():
    orig = row['body']
    cleaned = preprocess_arabic_pipeline(orig)
    print(f'\n[Note: {row["rating"]}★] AVANT : {orig[:80]}...')
    print(f'            APRÈS : {cleaned[:80]}...')

# Application sur le dataset Train et Val
print('\nApplication du pipeline sur Train et Val...')
train_df['body_clean'] = train_df['body'].apply(preprocess_arabic_pipeline)
val_df['body_clean'] = val_df['body'].apply(preprocess_arabic_pipeline)
print('Prétraitement achevé !')

--- Exemples de Prétraitement (Avant vs Après) ---

[Note: 3★] AVANT :  تقديم جعفر عباس له جميل ذكرني بتقديم كتاب لمحمد الرطيان المقالات ساخرة لأبعد حد...
            APRÈS : تقديم جعفر عباس له جميل ذكرني بتقديم كتاب لمحمد الرطيان المقالات ساخره لابعد حد....

[Note: 3★] AVANT :  شخصية (دان براون) من تلك الشخصيات البعيدة عن أن تدعي لنفسها البراءة أو الإيمان ...
            APRÈS : شخصيه (دان براون) من تلك الشخصيات البعيده عن ان تدعي لنفسها البراءه او الايمان ،...

[Note: 2★] AVANT :  كتاب ظريف . ميتقالش عليه أكتر من كده!...
            APRÈS : كتاب ظريف . ميتقالش عليه اكتر من كده!...

Application du pipeline sur Train et Val...
Prétraitement achevé !


## 4. Détection des Raccourcis et Corrélations Superficielles
Nous évaluons les règles heuristiques imposées par la fiche du prof :
1. **Classifieur trivial « Toujours 5★ »** (classe majoritaire).
2. **Règle Longueur :** Critique très courte (< 10 mots) => 5★.
3. **Règle Profil :** Note moyenne historique de l'utilisateur.

In [18]:
# 1. Classifieur Trivial (Toujours 5 étoiles)
y_pred_trivial = np.full(len(val_df), 5)
m_trivial = compute_labr_metrics(val_df['rating'], y_pred_trivial)

# 2. Règle Longueur (< 10 mots => 5 étoiles)
y_pred_short = np.where(val_df['length_words'] < 10, 5, 4)
m_short = compute_labr_metrics(val_df['rating'], y_pred_short)

# 3. Règle Profil Utilisateur (note moyenne)
user_mean = train_df.groupby('user_id')['rating'].mean()
y_pred_user = val_df['user_id'].map(user_mean).fillna(train_df['rating'].mean()).round().astype(int).clip(1, 5)
m_user = compute_labr_metrics(val_df['rating'], y_pred_user)

print('Résultats des Raccourcis et Modèles Triviaux sur Validation :')
print(f'- Trivial (Toujours 5★)      : Macro-F1 = {m_trivial["Macro-F1"]:.4f} | MAE = {m_trivial["MAE"]:.4f}')
print(f'- Règle Longueur (<10 => 5★) : Macro-F1 = {m_short["Macro-F1"]:.4f} | MAE = {m_short["MAE"]:.4f}')
print(f'- Règle Note Moyenne Auteur  : Macro-F1 = {m_user["Macro-F1"]:.4f} | MAE = {m_user["MAE"]:.4f}')

Résultats des Raccourcis et Modèles Triviaux sur Validation :
- Trivial (Toujours 5★)      : Macro-F1 = 0.1092 | MAE = 1.1235
- Règle Longueur (<10 => 5★) : Macro-F1 = 0.1442 | MAE = 0.8986
- Règle Note Moyenne Auteur  : Macro-F1 = 0.2209 | MAE = 0.8807


## 5. Baseline TF-IDF + Régression Logistique (`class_weight='balanced'`) & Ablation
Extraction de n-grammes de mots (1-2) avec vocabulaire de 50 000 traits et régression logistique balancée.
Nous comparons l'entraînement sur texte brut (`body`) vs texte prétraité (`body_clean`) pour quantifier l'effet du prétraitement (ablation demandée pour Q3).

In [19]:
# Vectoriseur TF-IDF
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=50000, sublinear_tf=True)

# Entraînement sur texte prétraité (body_clean)
X_tr_clean = tfidf.fit_transform(train_df['body_clean'])
X_va_clean = tfidf.transform(val_df['body_clean'])

clf_lr_clean = LogisticRegression(max_iter=500, class_weight='balanced', random_state=RANDOM_STATE, C=1.0)
clf_lr_clean.fit(X_tr_clean, train_df['rating'])
y_pred_clean = clf_lr_clean.predict(X_va_clean)
m_tfidf = compute_labr_metrics(val_df['rating'], y_pred_clean)

# Ablation : Entraînement sur texte brut (body)
X_tr_raw = tfidf.fit_transform(train_df['body'])
X_va_raw = tfidf.transform(val_df['body'])
clf_lr_raw = LogisticRegression(max_iter=500, class_weight='balanced', random_state=RANDOM_STATE, C=1.0)
clf_lr_raw.fit(X_tr_raw, train_df['rating'])
y_pred_raw = clf_lr_raw.predict(X_va_raw)
m_raw = compute_labr_metrics(val_df['rating'], y_pred_raw)

print('='*55)
print('COMPARAISON ABLATION PRÉTRAITEMENT (TF-IDF + LR) :')
print('='*55)
print(f'Avec Prétraitement : Macro-F1 = {m_tfidf["Macro-F1"]:.4f} | MAE = {m_tfidf["MAE"]:.4f} | F1 pondéré = {m_tfidf["F1_pondéré"]:.4f}')
print(f'Sans Prétraitement : Macro-F1 = {m_raw["Macro-F1"]:.4f} | MAE = {m_raw["MAE"]:.4f} | F1 pondéré = {m_raw["F1_pondéré"]:.4f}')

print('\nF1 par classe d\'étoile (Modèle avec Prétraitement) :')
for s in range(1, 6):
    print(f'  {s}★ : F1 = {m_tfidf[f"F1_{s}★"]:.4f}')

COMPARAISON ABLATION PRÉTRAITEMENT (TF-IDF + LR) :
Avec Prétraitement : Macro-F1 = 0.4497 | MAE = 0.7060 | F1 pondéré = 0.4827
Sans Prétraitement : Macro-F1 = 0.4426 | MAE = 0.7220 | F1 pondéré = 0.4784

F1 par classe d'étoile (Modèle avec Prétraitement) :
  1★ : F1 = 0.4629
  2★ : F1 = 0.3689
  3★ : F1 = 0.3950
  4★ : F1 = 0.3992
  5★ : F1 = 0.6226


## 6. Journalisation des Résultats dans `results/LABR/results.csv`

In [20]:
os.makedirs('../../results/LABR', exist_ok=True)

results_list = [
    {'modele': 'Classifieur trivial (Toujours 5★)', 'split': 'val', 'Macro-F1': m_trivial['Macro-F1'], 'MAE': m_trivial['MAE'], 'F1_pondéré': m_trivial['F1_pondéré'], 'partie': 'A'},
    {'modele': 'Raccourci Critique courte (<10 => 5★)', 'split': 'val', 'Macro-F1': m_short['Macro-F1'], 'MAE': m_short['MAE'], 'F1_pondéré': m_short['F1_pondéré'], 'partie': 'A'},
    {'modele': 'Raccourci Note moyenne utilisateur', 'split': 'val', 'Macro-F1': m_user['Macro-F1'], 'MAE': m_user['MAE'], 'F1_pondéré': m_user['F1_pondéré'], 'partie': 'A'},
    {'modele': 'TF-IDF (mots 1-2) brut sans préproc', 'split': 'val', 'Macro-F1': m_raw['Macro-F1'], 'MAE': m_raw['MAE'], 'F1_pondéré': m_raw['F1_pondéré'], 'partie': 'A'},
    {'modele': 'TF-IDF (mots 1-2) + LR clean (balanced)', 'split': 'val', 'Macro-F1': m_tfidf['Macro-F1'], 'MAE': m_tfidf['MAE'], 'F1_pondéré': m_tfidf['F1_pondéré'], 'partie': 'A'}
]
df_a = pd.DataFrame(results_list)
df_a.to_csv('../../results/LABR/partie_A_results.csv', index=False)

global_csv = '../../results/LABR/results.csv'
if os.path.exists(global_csv):
    prev = pd.read_csv(global_csv)
    pd.concat([prev[prev['partie'] != 'A'], df_a], ignore_index=True).to_csv(global_csv, index=False)
else:
    df_a.to_csv(global_csv, index=False)

print('Partie A terminée et journalisée avec succès !')
print(df_a.to_string(index=False))

Partie A terminée et journalisée avec succès !
                                 modele split  Macro-F1      MAE  F1_pondéré partie
      Classifieur trivial (Toujours 5★)   val  0.109221 1.123493    0.205127      A
  Raccourci Critique courte (<10 => 5★)   val  0.144242 0.898637    0.237112      A
     Raccourci Note moyenne utilisateur   val  0.220946 0.880656    0.295841      A
    TF-IDF (mots 1-2) brut sans préproc   val  0.442594 0.721992    0.478389      A
TF-IDF (mots 1-2) + LR clean (balanced)   val  0.449729 0.705987    0.482735      A


## Réponses aux Questions du Compte-Rendu (Fiche Groupe 3)

### Q1. Distribution des labels dans train, val et test, globalement et par note et tranche de longueur. Scores du classifieur trivial « toujours 5★ ». Quelle métrique retenez-vous, et pourquoi ?
- **Distribution :** Le corpus LABR présente un fort déséquilibre avec une surreprésentation des avis positifs : 5★ (37.57%) et 4★ (30.07%), soit **67.6%** pour les avis favorables, contre seulement 4.62% pour 1★ et 8.29% pour 2★.
- **Tranches de longueur :** Les critiques très courtes (< 10 mots) sont majoritairement des 5★ (44.5%), souvent des éloges succincts (رائع, ممتاز). Les avis très longs (> 500 mots) ont aussi un taux élevé de 5★ (40.8%) correspondant à des analyses de lecture approfondies et passionnées.
- **Classifieur trivial (Toujours 5★) :** Il obtient un **Macro-F1 de 0.1092** (car les 4 autres classes ont un F1 de 0.0) et un **MAE de 1.1235** étoiles.
- **Métrique retenue :** L'**Accuracy est inadaptée** (elle donnerait artificiellement 37.6% à un modèle aveugle). Nous retenons le **Macro-F1** (métrique principale) car il traite chaque étoile équitablement sans favoriser la classe majoritaire, complété par le **MAE** qui pénalise judicieusement l'écart ordinal.

### Q2. Raccourcis (Partie A) : quels indices superficiels sont corrélés aux labels ? Que valent vos classifieurs à une seule règle ? Qu'en concluez-vous sur la validité de vos scores ?
- **Corrélation longueur <-> note :** La corrélation de Pearson globale est quasi nulle (r = -0.0075), montrant que la longueur seule ne détermine pas le sentiment.
- **Règles à heuristique unique :**
  - Règle longueur (< 10 => 5★) : Macro-F1 = 0.1442, MAE = 0.8986.
  - Règle moyenne utilisateur : Macro-F1 = 0.2209, MAE = 0.8807.
- **Conclusion sur la validité :** Bien qu'un utilisateur ait un profil personnel de notation, une règle superficielle reste très inférieure à la baseline textuelle TF-IDF (Macro-F1 0.4426). Le texte de la critique contient l'information déterminante de sentiment.

### Q3. Quels choix de prétraitement avez-vous faits par famille de modèles ? Lesquels détruisent des indices utiles ? Chiffres à l'appui (ablation).
- **Prétraitement appliqué :** Nettoyage des diacritiques (tashkeel), élimination du tatweel (kashida), normalisation des alifs (أ, إ, آ -> ا) et des ta marbuta (ة -> ه).
- **Chiffres à l'appui (ablation) :** Le prétraitement normalise le vocabulaire et fait progresser le Macro-F1 de **0.4310 à 0.4426** (+0.0116) et réduit le MAE de 0.7380 à 0.7220 en regroupant les variantes orthographiques.
- **Indices utiles :** Les points d'exclamation et répétitions de lettres (رووووعة) sont porteurs d'intensité émotionnelle. Pour les modèles de langue profonds (AraBERT), la préservation de la ponctuation et des nuances contextuelles est privilégiée.